## Lets start building 

In [1]:
using Random
using Statistics
using LinearAlgebra
using Printf

# Set a seed for reproducibility of random weights
Random.seed!(42)

TaskLocalRNG()

In [2]:
mutable struct Layer_dense
    inputs::Matrix{Float64}
    weights::Matrix{Float64} 
    biases ::Matrix{Float64}
    output::Matrix{Float64} 
    dweights::Matrix{Float64}
    dbiases::Matrix{Float64} # Corrected from dbaises
    dinputs::Matrix{Float64}

    function Layer_dense(n_inputs::Int,n_neurons::Int)
        # Initialize weights with small random numbers from a Gaussian distribution
        weights=0.01*randn(n_inputs,n_neurons)
        biases=zeros(Float64,1,n_neurons)
        new(Matrix{Float64}(undef,0,0),weights,biases,Matrix{Float64}(undef,0,0),Matrix{Float64}(undef,0,0),Matrix{Float64}(undef,0,0),Matrix{Float64}(undef,0,0))
    end
        
end

In [4]:
# Forward Pass for dense layer
function forward(layer::Layer_dense,inputs::Matrix{Float64})
    # Store inputs for use in the backward pass
    layer.inputs = inputs 
    # Calculate output: Z = XW + B
    # Note: .+ is broadcasted addition for the bias vector
    layer.output=inputs*layer.weights.+layer.biases

end

forward (generic function with 1 method)

In [5]:
layer = Layer_dense(3,2)

Layer_dense(Matrix{Float64}(undef, 0, 0), [0.007883556016042917 -0.007332549644348927; -0.008798585959543994 -0.007021951987576804; -0.00873793482209626 -0.0007258186804586992], [0.0 0.0], Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0))

In [6]:
layer.weights

3×2 Matrix{Float64}:
  0.00788356  -0.00733255
 -0.00879859  -0.00702195
 -0.00873793  -0.000725819

In [7]:
layer.inputs

0×0 Matrix{Float64}

In [8]:
layer.biases

1×2 Matrix{Float64}:
 0.0  0.0

In [9]:
# Mock data: 4 samples, each with 3 features
X = [1.0 2.0 3.0;
     4.0 5.0 6.0;
     7.0 8.0 9.0;
     1.0 1.0 1.0
     ]

4×3 Matrix{Float64}:
 1.0  2.0  3.0
 4.0  5.0  6.0
 7.0  8.0  9.0
 1.0  1.0  1.0

In [10]:
X_single = [1.0 2.0 3.0]

1×3 Matrix{Float64}:
 1.0  2.0  3.0

In [11]:
forward(layer,X_single)

1×2 Matrix{Float64}:
 -0.0359274  -0.0235539

In [12]:
forward(layer,X)

4×2 Matrix{Float64}:
 -0.0359274   -0.0235539
 -0.0648863   -0.0687949
 -0.0938452   -0.114036
 -0.00965296  -0.0150803

In [13]:
function backward(layer::Layer_dense,upsteam_gradient::Matrix{Float64})
    # Gradient of the loss with respect to weights: dL/dW = X^T * dL/dZ
    # In Julia, ' is the adjoint (transpose for real matrices)
    layer.dweights=layer.inputs'*upsteam_gradient
      # Gradient of the loss with respect to biases: dL/dB = sum(dL/dZ)
     layer.dbiases = sum(upsteam_gradient, dims=1)
     # Gradient of the loss with respect to inputs: dL/dX = dL/dZ * W^T
     layer.dinputs=upsteam_gradient*layer.weights'

    
end

backward (generic function with 1 method)

In [14]:
upstream_gradient = [0.1 0.2;
                     0.3 0.4;
                     0.5 0.6;
                     0.7 0.8]

4×2 Matrix{Float64}:
 0.1  0.2
 0.3  0.4
 0.5  0.6
 0.7  0.8

In [15]:
X = [1.0 2.0 3.0;
     4.0 5.0 6.0;
     7.0 8.0 9.0;
     1.0 1.0 1.0]

4×3 Matrix{Float64}:
 1.0  2.0  3.0
 4.0  5.0  6.0
 7.0  8.0  9.0
 1.0  1.0  1.0

In [16]:
layer = Layer_dense(3, 2)

Layer_dense(Matrix{Float64}(undef, 0, 0), [-0.009129233863399266 -0.0035416984337044606; 0.006316208311167526 0.00796126919278033; 0.014386832757114134 -0.01175371133217587], [0.0 0.0], Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0))

In [17]:
forward_output=forward(layer, X)

4×2 Matrix{Float64}:
 0.0466637  -0.0228803
 0.0813851  -0.0448827
 0.116107   -0.0668851
 0.0115738  -0.00733414

In [18]:
upstream_gradient = [0.1 0.2;
                     0.3 0.4;
                     0.5 0.6;
                     0.7 0.8]

4×2 Matrix{Float64}:
 0.1  0.2
 0.3  0.4
 0.5  0.6
 0.7  0.8

In [19]:
# ----------------------------------------------------
# 4. Backward Pass Test
# ----------------------------------------------------
Backward_output=backward(layer, upstream_gradient)

4×3 Matrix{Float64}:
 -0.00162126  0.00222387  -0.000912059
 -0.00415545  0.00507937  -0.000385435
 -0.00668964  0.00793487   0.00014119
 -0.00922382  0.0107904    0.000667814

In [20]:
layer.dweights

3×2 Matrix{Float64}:
 5.5  6.8
 6.4  8.0
 7.3  9.2

In [21]:
layer.dbaises

ErrorException: type Layer_dense has no field dbaises

In [22]:
layer.dinputs

4×3 Matrix{Float64}:
 -0.00162126  0.00222387  -0.000912059
 -0.00415545  0.00507937  -0.000385435
 -0.00668964  0.00793487   0.00014119
 -0.00922382  0.0107904    0.000667814

In [23]:
size(forward_output)

(4, 2)

## Lets pass it through the ReLU activation 

In [24]:
mutable struct Activation_ReLU
    inputs::Matrix{Float64}
    output::Matrix{Float64}
    dinputs::Matrix{Float64}
    Activation_ReLU()=new(Matrix{Float64}(undef,0,0), Matrix{Float64}(undef,0,0), Matrix{Float64}(undef,0,0))
end

In [25]:
# Forward pass for ReLU
function forward(activation::Activation_ReLU,inputs::Matrix{Float64})
    # Store inputs for the backward pass
    activation.inputs = inputs
    # Apply ReLU: max(0, input)
    # The dot . applies the function element-wise
    activation.output = max.(0, inputs)
    
end


forward (generic function with 2 methods)

In [26]:
# Backward pass for ReLU
function backward(activation::Activation_ReLU, upstream_gradient::Matrix{Float64})
    # Start with a copy of the upstream gradient
    activation.dinputs = copy(upstream_gradient)
    # Zero out gradients where the original input was non-positive
    # This creates a boolean mask and applies it
    activation.dinputs[activation.inputs .<= 0] .= 0.0f0
end

backward (generic function with 2 methods)

In [27]:
relu_layer = Activation_ReLU()

Activation_ReLU(Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0))

In [28]:
Z_input = [ 0.5 -0.2;
           -1.0  0.0;
            2.3  1.1;
           -0.8  0.6]

4×2 Matrix{Float64}:
  0.5  -0.2
 -1.0   0.0
  2.3   1.1
 -0.8   0.6

In [29]:
forward(relu_layer,Z_input)

4×2 Matrix{Float64}:
 0.5  0.0
 0.0  0.0
 2.3  1.1
 0.0  0.6

In [30]:
println("--- ReLU Forward Pass ---")
println("Input (Z):")
println(relu_layer.inputs)

--- ReLU Forward Pass ---
Input (Z):
[0.5 -0.2; -1.0 0.0; 2.3 1.1; -0.8 0.6]


In [31]:
println(relu_layer.output)

[0.5 0.0; 0.0 0.0; 2.3 1.1; 0.0 0.6]


In [32]:
upstream_grad = [ 0.1  0.2;
                  0.3  0.4;
                  0.5  0.6;
                  0.7  0.8]

4×2 Matrix{Float64}:
 0.1  0.2
 0.3  0.4
 0.5  0.6
 0.7  0.8

In [33]:
backward(relu_layer,upstream_grad)

4-element view(::Vector{Float64}, [2, 4, 5, 6]) with eltype Float64:
 0.0
 0.0
 0.0
 0.0

In [35]:
# 5. Perform Backward Pass
backward(relu_layer, upstream_grad)

println("\n--- ReLU Backward Pass ---")
println("Upstream Gradient (dL/dA):")
println(upstream_grad)

println("\nGradient to Previous Layer (dL/dZ):")
println(relu_layer.dinputs)


--- ReLU Backward Pass ---
Upstream Gradient (dL/dA):
[0.1 0.2; 0.3 0.4; 0.5 0.6; 0.7 0.8]

Gradient to Previous Layer (dL/dZ):
[0.1 0.0; 0.0 0.0; 0.5 0.6; 0.0 0.8]


In [36]:
relu_layer.dinputs

4×2 Matrix{Float64}:
 0.1  0.0
 0.0  0.0
 0.5  0.6
 0.0  0.8

In [37]:
X = [1.0 2.0 3.0;
     4.0 5.0 6.0;
     7.0 8.0 9.0;
     1.0 1.0 1.0]

4×3 Matrix{Float64}:
 1.0  2.0  3.0
 4.0  5.0  6.0
 7.0  8.0  9.0
 1.0  1.0  1.0

In [38]:
layer_dense=Layer_dense(3,2)

Layer_dense(Matrix{Float64}(undef, 0, 0), [-0.005933950393067663 0.0012913921166034415; 0.0042960636497195205 -0.0029477389356113567; 0.009523284631212728 -0.0037426843521845446], [0.0 0.0], Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0))

In [39]:
layer_relu=Activation_ReLU()

Activation_ReLU(Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0), Matrix{Float64}(undef, 0, 0))

In [40]:
println("--- Initialization Complete ---")
println("Input data shape [x] : ",size(X))
println("weights (W) : ", size(layer_dense.weights))
# Note: layer_dense.weights will have random non-zero values here.


--- Initialization Complete ---
Input data shape [x] : (4, 3)
weights (W) : (3, 2)


In [41]:
forward(layer_dense,X)
z=layer_dense.output
println("\n--- Layer 1: Dense Forward Pass (Z) ---")
println("Shape of Z (layer_dense.output): ", size(z))
println(z)


--- Layer 1: Dense Forward Pass (Z) ---
Shape of Z (layer_dense.output): (4, 2)
[0.031228030800009562 -0.015832138811172906; 0.05488422446360332 -0.03202923232475029; 0.07854041812719707 -0.048226325838327665; 0.007885397887864586 -0.00539903117119246]


In [42]:
A=forward(layer_relu,z)

4×2 Matrix{Float64}:
 0.031228   0.0
 0.0548842  0.0
 0.0785404  0.0
 0.0078854  0.0

In [43]:
layer_relu.inputs

4×2 Matrix{Float64}:
 0.031228   -0.0158321
 0.0548842  -0.0320292
 0.0785404  -0.0482263
 0.0078854  -0.00539903

In [44]:
layer_relu.output


4×2 Matrix{Float64}:
 0.031228   0.0
 0.0548842  0.0
 0.0785404  0.0
 0.0078854  0.0

In [45]:
println("\n--- Layer 2: ReLU Forward Pass (A) ---")
println("Shape of A (layer_relu.output): ", size(A))
println(A)


--- Layer 2: ReLU Forward Pass (A) ---
Shape of A (layer_relu.output): (4, 2)
[0.031228030800009562 0.0; 0.05488422446360332 0.0; 0.07854041812719707 0.0; 0.007885397887864586 0.0]


In [47]:
# Get the gradient calculated by the ReLU layer (dL/dZ)
gradient_from_relu = [ 0.1 0.0;
                       0.0 0.0;
                       0.5 0.6;
                        0.0 0.8]  # -----> dL/dZ

4×2 Matrix{Float64}:
 0.1  0.0
 0.0  0.0
 0.5  0.6
 0.0  0.8

In [48]:
backward(layer_dense,gradient_from_relu)

4×3 Matrix{Float64}:
 -0.000593395   0.000429606   0.000952328
  0.0           0.0           0.0
 -0.00219214    0.000379388   0.00251603
  0.00103311   -0.00235819   -0.00299415

In [ ]:
println("--- Final Gradients for Layer_dense ---")
println("1. dWeights (dL/dW) for Layer_dense:")
println(layer_dense.dweights)

--- Final Gradients for Layer_dense ---
1. dWeights (dL/dW) for Layer_dense:
[3.6 5.0; 4.2 5.6; 4.8 6.199999999999999]


In [50]:
println("--- Final Gradients for Layer_dense ---")
println("1. dWeights (dL/dW) for Layer_dense:")
println(layer_dense.dweights)  # Used to update W

println("\n2. dBiases (dL/dB) for Layer_dense:")
println(layer_dense.dbiases)   # Used to update B

println("\n3. dInputs (dL/dX) to pass upstream (to the input X):")
println(layer_dense.dinputs)

--- Final Gradients for Layer_dense ---
1. dWeights (dL/dW) for Layer_dense:
[3.6 5.0; 4.2 5.6; 4.8 6.199999999999999]

2. dBiases (dL/dB) for Layer_dense:
[0.6 1.4]

3. dInputs (dL/dX) to pass upstream (to the input X):
[-0.0005933950393067663 0.0004296063649719521 0.0009523284631212728; 0.0 0.0 0.0; -0.0021921399265717664 0.0003793884634929463 0.002516031704295637; 0.0010331136932827533 -0.0023581911484890855 -0.0029941474817476357]


## Lets move to Activation_Softmax_Loss_CategoricalCrossentropy (Combined)

In [51]:
mutable struct Activation_Softmax_Loss_CategoricalCrossentropy
    output::Matrix{Float64} # This will store the softmax probabilities
    dinputs::Matrix{Float64}
    
    Activation_Softmax_Loss_CategoricalCrossentropy() = new(Matrix{Float64}(undef,0,0), Matrix{Float64}(undef,0,0))
end


In [52]:
function forward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, inputs::Matrix{Float32}, y_true::Vector{Int})
     exp_values = exp.(inputs .- maximum(inputs, dims=2))
    probabilities = exp_values ./ sum(exp_values, dims=2)
    combo.output = probabilities
    # 2. Categorical Cross-Entropy Loss Calculation
    n_samples = size(probabilities, 1)
        # Clip data to prevent division by 0
    probabilities_clipped = clamp.(probabilities, 1e-7, 1 - 1e-7)
    # Get the probabilities corresponding to the true labels
    # Note: Julia is 1-indexed, so labels must be 1, 2, or 3
    correct_confidences = [probabilities_clipped[i, y_true[i]] for i in 1:n_samples]
        # Calculate negative log likelihoods and return the average loss
    negative_log_likelihoods = -log.(correct_confidences)
    data_loss = mean(negative_log_likelihoods)
    return data_loss

end


forward (generic function with 3 methods)

In [ ]:
mutable struct Optimizer_SGD
    


end